# Data Merging — Stage 3 Cleaning 01: Apply Exclusions

## Input
The four Stage 2 aggregate tables:
- `Data/Data_Collection/Final/Stage_2/agg_market_daily_means.parquet`
- `Data/Data_Collection/Final/Stage_2/agg_market_daily_full_moments.parquet`
- `Data/Data_Collection/Final/Stage_2/agg_market_monthly_means.parquet`
- `Data/Data_Collection/Final/Stage_2/agg_market_monthly_full_moments.parquet`

## Purpose
The first notebook in Stage 3 Cleaning. Applies a single, centrally-defined exclusion list to drop factors (or specific moments of factors) from all four aggregate tables before normalisation. Every exclusion carries a category and a written reason, which are written out to a manifest for the eventual writeup rather than justified only in code comments.

## Design Notes

**Naming convention:** Stage 2 monthly factors are unprefixed at this point (`monthly_` is only added later when daily and monthly tables are combined). A drop-list entry `X` therefore expands to `X` in the means tables and `X_cwmean`, `X_cwstd`, `X_cwskew`, `X_cwkurt`, `X_spread` in the full-moments tables, and only the moments actually present in each table are removed. Column names are generated from the drop list rather than pattern-matched out of the data, so factor names that happen to end in `_spread` without being spread moments (`lev_spread`, `bull_bear_spread`) are handled safely.

**No transform notebook:** An earlier plan to add a stationarity-transform notebook after this one was abandoned. The reasoning: an expanding-window z-score (used in Stage 4) already neutralises monotone trends on its own — a linear trend produces a constant z of ≈1.73 regardless of slope, and exponential growth reaches z ≈ 3.7 even for a 1,000-fold rise. What the stationarity scan actually surfaced was **redundancy** (dollar-denominated factors with stationary percent twins, levels with existing `_mom` variants already retained, degenerate moments with zero cross-sectional variance) — these are handled here as drops, not as things needing a transform.

## Exclusion Categories

1. **Calendar** — bounded 0/1 or ordinal calendar indicators (day-of-week, month-of-year, OPEX week, etc.) that should never be z-scored.
2. **Broken TAQ counts** — interval-count factors (`n30_pos`, `n5_pos`, `n_obs`) that are implausibly constant across most stock-days (cwstd exactly zero on 67–97% of dates).
3. **Zero information** — factors with near-constant or degenerate values (`vxd_overnight_gap`, `rf` during ZIRP).
4. **Redundant dollar spreads** — an entire family of dollar-denominated TAQ spread/impact factors that rise 4–6x across the sample purely because the index level rose ~4x, verified against a `TWINS` mapping to their stationary percent-denominated equivalents (checked at the end to guarantee no measurement is lost entirely).
5. **Redundant weighting** — dollar/share-weighted (`_dw`/`_sw`) variants of spread factors dropped in favour of the stable equal-weighted (`_ave`) variant.
6. **Redundant venue** — per-venue intraday ranges superseded by the all-venue CRSP range.
7. **Decided** — case-by-case drops for specific broken OAP factors (`HerfBE`, `ChInvIA`) with extreme outlier values.
8. **Level superseded** — non-stationary money-supply levels (`m1`, `m2`, `monetary_base`) dropped in favour of already-retained `_mom` variants.
9. **Weekly rebuild** — all CFTC, AAII, and FRED-weekly factors, currently forward-filled to daily (which caused drift under z-scoring). Removed here to be rebuilt at native weekly cadence in notebook 02.
10. **Degenerate moment / sparse indicator** — partial drops that keep a factor's `cwmean` but remove `cwstd`/`spread` (and previously-dropped `cwskew`/`cwkurt`) where the cross-sectional dispersion is exactly zero across most of the sample (e.g. dividend-only returns, rare corporate-event indicators).

## Validation
- **Twin check:** for every dollar factor dropped via a `TWINS` mapping, asserts its percent-denominated twin is actually present in the output — raises if a measurement would be lost entirely with no replacement.
- **Typo check:** flags (without failing) any drop-list entry that matched zero columns across all four tables, since every entry should match something somewhere.
- Per-table before/after column counts and a per-category breakdown are printed for each of the four files.

## Output
- `Data/Data_Collection/Final/Stage_3_Cleaning/agg_market_daily_means.parquet`
- `Data/Data_Collection/Final/Stage_3_Cleaning/agg_market_daily_full_moments.parquet`
- `Data/Data_Collection/Final/Stage_3_Cleaning/agg_market_monthly_means.parquet`
- `Data/Data_Collection/Final/Stage_3_Cleaning/agg_market_monthly_full_moments.parquet`
- `Data/Data_Collection/Final/Stage_3_Cleaning/exclusion_manifest.csv` — one row per dropped column, with factor, category, and reason, for use in the writeup.

In [3]:
"""
Stage 3 Cleaning — 01: Apply Exclusions
=======================================
Drops factors from the four Stage 2 aggregate tables.

One list, one loop. Every entry carries a reason, and the reasons become a table
in the writeup.

Input:   Data/Data_Collection/Final/Stage_2/*.parquet
Output:  Data/Data_Collection/Final/Stage_3_Cleaning/*.parquet
         exclusion_manifest.csv

NAMING
------
Stage 2 tables carry monthly factors UNPREFIXED -- the monthly_ prefix is only
added later, when daily and monthly are combined. So the entry is HerfBE, not
monthly_HerfBE.

A factor named X appears as:
    X                                      in the means tables
    X_cwmean, X_cwstd, X_cwskew,           in the full-moments tables
    X_cwkurt, X_spread
Dropping X removes whichever of those exist.

Some factor NAMES end in _spread without being spread moments (lev_spread,
other_spread, bull_bear_spread). Harmless here, because column names are
GENERATED from the drop list rather than parsed out of the data.

NO TRANSFORMS AFTER THIS
------------------------
The planned 03b transform notebook was dropped. Working through the arithmetic,
a monotone trend does NOT produce extreme z-scores: a pure linear trend gives a
constant z of sqrt(3) = 1.73 regardless of slope, and exponential growth gives
z ~ sqrt(2 * ln G), so even a 1,000-fold rise reaches only 3.7. An expanding
z-score effectively detrends the series on its own.

What the stationarity scan actually found was REDUNDANCY -- dollar-denominated
factors with percent twins, levels with existing _mom variants, bond indices
that are cumulative products of return series already held. Those are drops,
handled here, and they need no z-scores to justify.

Anything still drifting after normalisation is caught in Stage 4's review, which
measures shift_max on the Z-SCORES directly rather than proxying it from the raw
level, and adds std_of_z to catch the opposite failure -- a feature flattened to
a near-constant with no variation left for a spline to read.
"""

import pandas as pd
from pathlib import Path

IN_DIR = Path('../../../Data/Data_Collection/Final/Stage_2')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_3_Cleaning')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MOMENTS = ['cwmean', 'cwstd', 'cwskew', 'cwkurt', 'spread']

TABLES = [
    'agg_market_daily_means.parquet',
    'agg_market_daily_full_moments.parquet',
    'agg_market_monthly_means.parquet',
    'agg_market_monthly_full_moments.parquet',
]

# Dropped -> retained. Checked after the drop: losing a dollar version is only
# safe if its percent twin survives, otherwise the measurement is gone entirely.
TWINS = {
    'effectivespread_dollar_ave':  'effectivespread_percent_ave',
    'dollarpriceimpact_lr_ave':    'percentpriceimpact_lr_ave',
    'dollarrealizedspread_lr_ave': 'percentrealizedspread_lr_ave',
    'quotedspread_dollar_tw':      'quotedspread_percent_tw',
}


# ═══════════════════════════════════════════════════════════════════════════════
# THE DROP LIST
# ═══════════════════════════════════════════════════════════════════════════════
# factor -> (moments, category, reason)
#   moments = None  : drop the factor entirely
#   moments = [...] : drop only those moments, keep the rest

DROP = {}


def add(factors, moments, category, reason):
    for f in factors:
        DROP[f] = (moments, category, reason)


# ── Calendar (Panel C) ──────────────────────────────────────────────────────
add(['day_of_week', 'is_monday', 'is_friday', 'month_of_year',
     'is_quarter_end', 'trading_days_to_month_end', 'is_turn_of_month',
     'is_opex_week'],
    None, 'calendar',
    'Bounded calendar indicator, never z-scored.')

# ── Broken TAQ counts (Panel A) ─────────────────────────────────────────────
add(['n30_pos'], None, 'broken_taq',
    'Count of 30-second intervals with positive midpoint returns. Identical on '
    '99.4% of stock-days; cwstd exactly zero on 97.4% of dates. A count over '
    '~780 intervals cannot legitimately be that constant.')
add(['n5_pos'], None, 'broken_taq',
    'Count of 5-minute intervals with positive returns. Identical on 98.4% of '
    'stock-days; cwstd zero on 69.9% of dates.')
add(['n_obs'], None, 'broken_taq',
    'Count of intervals with valid quotes. A coverage diagnostic rather than a '
    'predictor; cwstd zero on 67.1% of dates.')

# ── Zero information (Panel C) ──────────────────────────────────────────────
add(['vxd_overnight_gap'], None, 'zero_information',
    'Exactly zero on 64.2% of days -- the VXD open equals the previous close, '
    'which is impossible for a real index and indicates a stale open price.')
add(['rf'], None, 'zero_information',
    'Daily risk-free rate: three distinct values across the sample, exactly '
    'zero on 63.3% of days (ZIRP).')

# ── Dollar-denominated spreads (Panel A) ────────────────────────────────────
# Whole family, all weightings. A dollar spread that is a constant FRACTION of
# price rises with price, and the S&P went from ~1,200 to ~5,000 over the
# sample. Percentage spreads narrowed over the same period on decimalisation
# and HFT; these rose because they are denominated in dollars.
#
#   quotedspread_dollar_tw       0.0159 -> 0.0340 -> 0.1365   shift_max 5.57
#   dollarrealizedspread_lr_ave  0.0039 -> 0.0048 -> 0.0259   shift_max 5.48
#   dollarpriceimpact_lr_ave     0.0104 -> 0.0192 -> 0.0424   shift_max 4.53
#   effectivespread_dollar_ave   0.0143 -> 0.0249 -> 0.0723   shift_max 4.40
#
# All four percent twins are retained and NONE appear in the scan's top 40, so
# the percent versions are stationary and the dollar ones are carrying the index
# level as a nuisance. Same logic as m1 versus m1_mom.
#
# Note this supersedes the _dw/_sw drop for these three families -- the entire
# family goes, so every column gets one consistent reason in the manifest.
add(['effectivespread_dollar_ave', 'effectivespread_dollar_dw',
     'effectivespread_dollar_sw',
     'dollarpriceimpact_lr_ave', 'dollarpriceimpact_lr_dw',
     'dollarpriceimpact_lr_sw',
     'dollarrealizedspread_lr_ave', 'dollarrealizedspread_lr_dw',
     'dollarrealizedspread_lr_sw',
     'quotedspread_dollar_tw'],
    None, 'redundant_dollar',
    'Dollar-denominated spread, rising 5-9x across the sample purely because '
    'the index level rose ~4x (shift_max 4.4-5.6). The percent twin measures '
    'the same quantity free of the price level and is retained.')

# ── Redundant weightings on the percent families (Panel A) ──────────────────
# _ave (equal-weighted) is retained. _dw divides by total dollar volume and _sw
# by total share volume, both of which approach zero on thin stock-days.
add([f'{fam}_{w}'
     for fam in ('effectivespread_percent', 'percentpriceimpact_lr',
                 'percentrealizedspread_lr')
     for w in ('dw', 'sw')],
    None, 'redundant_weighting',
    'Dollar- or share-weighted variant. The _ave variant measures the same '
    'quantity through a stable denominator: _dw shows std/sigma_f of 126-257 '
    'against 1.8-2.2 for _ave.')

# ── Redundant venue ranges (Panel A) ────────────────────────────────────────
add(['venue_range_a', 'venue_range_b', 'venue_range_m'],
    None, 'redundant_venue',
    'Per-venue intraday range. CRSP intraday_range measures the same quantity '
    'across all venues and is well behaved.')

# ── Decided (Panel B) ───────────────────────────────────────────────────────
add(['HerfBE'], None, 'decided',
    'Herfindahl on book equity, which approaches zero for some firms. Worst '
    'feature in the dataset by boundary mass (8.8-11.7%). Herf (sales) and '
    'HerfAsset (assets) measure the same concept on stable denominators.')
add(['ChInvIA'], None, 'decided',
    'Industry-adjusted inventory change. Contains a value of order -8.7e12, '
    'inflating the standard deviation to 4.1e11 times the robust scale.')

# ── Levels superseded by existing _mom variants (Panel D) ───────────────────
add(['m1', 'm2', 'monetary_base'], None, 'level_superseded',
    'Non-stationary level; the _mom variant already exists and is retained. M1 '
    'went from $4tn to $16tn in May 2020 on the savings-deposit '
    'reclassification, so the level is uninterpretable.')

# ── Weekly features (Panel C) -- rebuilt in notebook 02 ─────────────────────
# Currently forward-filled to daily, which is why z-scoring them at daily
# frequency drifted. Notebook 02 rebuilds them at native weekly cadence.
add(['lev_long', 'lev_short', 'lev_spread', 'am_long', 'am_short', 'am_spread',
     'dealer_long', 'dealer_short', 'dealer_spread', 'other_long',
     'other_short', 'other_spread', 'open_interest', 'lev_net', 'am_net',
     'dealer_net', 'lev_net_pct', 'am_net_pct', 'dealer_net_pct',
     'lev_am_ratio', 'lev_net_chg', 'am_net_chg',
     'bullish', 'neutral', 'bearish', 'bullish_8w_ma', 'bull_bear_spread',
     'initial_claims', 'continued_claims',
     'fed_assets', 'tga', 'reserves', 'bank_credit', 'ci_loans'],
    None, 'weekly_rebuild',
    'Weekly-sourced, forward-filled to daily. Removed here and rebuilt at '
    'native weekly frequency in notebook 02.')

# ── Partial drops: keep the cwmean, drop the degenerate moments ─────────────
add(['dlyreti'], ['cwstd', 'spread'], 'degenerate_moment',
    'Dividend-only return: cross-sectional dispersion is exactly zero on 39.4% '
    'of dates. The cwmean is a meaningful market dividend-yield series and is '
    'retained.')

add(['DivSeason', 'rec_median', 'analyst_alignment'], ['spread'],
    'degenerate_moment',
    'p90-p10 identical in every month of the sample. Zero variance.')

add(['earnings_surprise_chg_1m'], ['spread'], 'degenerate_moment',
    'p90-p10 exactly zero in 63.8% of months, meaning at least 80% of the '
    'cross-section was identical.')

add(['DivInit', 'DivOmit', 'Spinoff', 'IndIPO', 'ExchSwitch',
     'delbreadth_chg_1m'],
    ['cwstd', 'spread'], 'sparse_indicator',
    'Corporate-event indicator: all firms identical in most months, so cwstd '
    'and spread are exactly zero. cwskew/cwkurt were already dropped in Stage '
    '2. The cwmean is retained pending the post-z-score review.')


# ═══════════════════════════════════════════════════════════════════════════════
# APPLY
# ═══════════════════════════════════════════════════════════════════════════════

def columns_to_drop(cols):
    """Expand the drop list into concrete column names present in `cols`."""
    present = set(cols)
    hits = []
    for factor, (moments, category, reason) in DROP.items():
        names = ([factor] + [f'{factor}_{m}' for m in MOMENTS]
                 if moments is None
                 else [f'{factor}_{m}' for m in moments])
        hits += [(n, factor, category, reason) for n in names if n in present]
    return hits


print("=" * 78)
print("STAGE 3 CLEANING — 01: APPLY EXCLUSIONS")
print("=" * 78)

n_full = sum(1 for v in DROP.values() if v[0] is None)
print(f"\n  Drop list: {len(DROP)} factors "
      f"({n_full} entirely, {len(DROP) - n_full} partially)")

manifest = []
matched = set()
twin_fail = []

for fname in TABLES:
    path = IN_DIR / fname
    if not path.exists():
        print(f"\n  MISSING: {path}")
        continue

    df = pd.read_parquet(path)
    before = df.shape[1]

    hits = columns_to_drop(df.columns)
    matched.update(f for _, f, _, _ in hits)

    df = df.drop(columns=[n for n, _, _, _ in hits])
    df.to_parquet(OUT_DIR / fname, index=False, engine='pyarrow')

    manifest += [{'table': fname, 'column': n, 'factor': f,
                  'category': c, 'reason': r} for n, f, c, r in hits]

    # A dollar version is only safe to drop if its percent twin survives.
    surviving = set(df.columns)
    for dropped, twin in TWINS.items():
        was_dropped = any(h[1] == dropped for h in hits)
        if not was_dropped:
            continue
        # The twin appears bare in the means tables and suffixed in full moments
        present = twin in surviving or any(
            f'{twin}_{m}' in surviving for m in MOMENTS)
        if not present:
            twin_fail.append((fname, dropped, twin))

    by_cat = {}
    for _, _, c, _ in hits:
        by_cat[c] = by_cat.get(c, 0) + 1

    print(f"\n  {fname}")
    print(f"    {before} -> {df.shape[1]} columns ({len(hits)} dropped)")
    for c in sorted(by_cat):
        print(f"      {c:<22s} {by_cat[c]:>4d}")

pd.DataFrame(manifest).to_csv(OUT_DIR / 'exclusion_manifest.csv', index=False)
print(f"\n  Saved manifest: {len(manifest)} rows")

# ── Twin check ──────────────────────────────────────────────────────────────
if twin_fail:
    print(f"\n  ** {len(twin_fail)} dollar factors dropped WITHOUT their "
          f"percent twin surviving -- the measurement is lost entirely **")
    for t, d, w in twin_fail:
        print(f"    {t}: dropped {d}, twin {w} not present")
    raise AssertionError("Percent twin missing for a dropped dollar factor")
print(f"  Percent twins verified present for all {len(TWINS)} dollar drops")

# ── Typo check ──────────────────────────────────────────────────────────────
# An entry matching nothing in any table is almost certainly a misspelling --
# every factor on the list should exist somewhere.
never = sorted(set(DROP) - matched)
if never:
    print(f"\n  ** {len(never)} entries matched NOTHING in any table -- check "
          f"spelling **")
    for f in never:
        print(f"    {f}")
else:
    print(f"  Every drop-list entry matched at least one table")

print(f"\n  Output: {OUT_DIR}")
print(f"  Next: 02_weekly_dedup, then straight to Stage 4 normalisation.")
print(f"        No transform notebook -- see the docstring.")

STAGE 3 CLEANING — 01: APPLY EXCLUSIONS

  Drop list: 82 factors (71 entirely, 11 partially)

  agg_market_daily_means.parquet
    400 -> 334 columns (66 dropped)
      broken_taq                3
      calendar                  8
      redundant_dollar         10
      redundant_venue           3
      redundant_weighting       6
      weekly_rebuild           34
      zero_information          2

  agg_market_daily_full_moments.parquet
    1148 -> 998 columns (150 dropped)
      broken_taq                9
      calendar                  8
      degenerate_moment         2
      redundant_dollar         50
      redundant_venue          15
      redundant_weighting      30
      weekly_rebuild           34
      zero_information          2

  agg_market_monthly_means.parquet
    327 -> 322 columns (5 dropped)
      decided                   2
      level_superseded          3

  agg_market_monthly_full_moments.parquet
    1083 -> 1054 columns (29 dropped)
      decided               